# TipCat Pipeline Manager — Mouse Pads
Interactive notebook for managing mouse pad designs and running pipeline steps.

In [ ]:
import json
import subprocess
from google.cloud import storage
from datetime import datetime

# Configuration
CONFIG_NAME = "tipcat-mousepads"
GCP_PROJECT = "tipcat-automation"
GCS_BUCKET = "tipcat-mousepads"
CLOUD_RUN_JOB = "tipcat-mousepads-pipeline"
CLOUD_RUN_REGION = "us-central1"

storage_client = storage.Client(project=GCP_PROJECT)
bucket = storage_client.bucket(GCS_BUCKET)
print(f"✓ Connected to GCP project: {GCP_PROJECT}")
print(f"✓ GCS bucket: {GCS_BUCKET}")
print(f"✓ Config: {CONFIG_NAME}")
print(f"✓ Cloud Run job: {CLOUD_RUN_JOB}")

In [ ]:
def list_designs(prefix="designs/"):
    """List all PNG designs in GCS bucket."""
    blobs = list(bucket.list_blobs(prefix=prefix))
    designs = [b.name for b in blobs if b.name.lower().endswith(".png")]
    print(f"\n📁 Found {len(designs)} PNG designs:")
    for name in designs[:20]:
        print(f"  - {name}")
    if len(designs) > 20:
        print(f"  ... and {len(designs) - 20} more")
    return designs

print("\n=== Scanning for designs ===")
designs = list_designs()

In [ ]:
def refresh_inventory():
    """Scan GCS for designs and update product list JSON."""
    designs = list_designs()
    
    # Create product list from design filenames
    products = []
    for i, design_path in enumerate(designs, 1):
        filename = design_path.split('/')[-1].replace('.png', '')
        products.append({
            "sku": str(i),
            "design_name": filename,
            "gcs_path": design_path,
            "status": "pending"
        })
    
    # Write product list to GCS
    product_list = {
        "config": CONFIG_NAME,
        "generated_at": datetime.now().isoformat(),
        "total_designs": len(products),
        "products": products
    }
    
    blob = bucket.blob("output/product_list.json")
    blob.upload_from_string(
        json.dumps(product_list, indent=2),
        content_type="application/json"
    )
    
    print(f"\n✓ Updated product_list.json: {len(products)} designs")
    return product_list

# Refresh on startup
product_list = refresh_inventory()

In [ ]:
def run_step(step=1, limit=None, verbose=True, config=CONFIG_NAME):
    """Execute a single pipeline step via Cloud Run."""
    args = [
        "--config",
        config,
        "--step",
        str(step)
    ]
    
    if limit and int(limit) > 0:
        args.extend(["--limit", str(int(limit))])
    
    if verbose:
        args.append("--verbose")
    
    cmd = [
        "gcloud", "run", "jobs", "execute", CLOUD_RUN_JOB,
        f"--region={CLOUD_RUN_REGION}",
        f"--project={GCP_PROJECT}",
        f"--args={','.join(args)}"
    ]
    
    print(f"\n🚀 Running Step {step}...")
    print(f"Command: {' '.join(cmd)}")
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(f"Exit code: {result.returncode}")
    
    if result.stdout:
        print("\n📋 Output:")
        print(result.stdout[:2000])
    
    if result.stderr:
        print("\n❌ Errors:")
        print(result.stderr[:2000])
    
    if result.returncode == 0:
        print(f"\n✓ Step {step} completed successfully")
    
    return result

# Example usage:
# result = run_step(step=1, limit=2, verbose=True)

In [ ]:
def read_generated_metadata():
    """Read and display generated metadata from Step 1."""
    try:
        blob = bucket.blob("output/generated_metadata.json")
        data = json.loads(blob.download_as_text())
        print(f"\n📊 Generated Metadata: {len(data)} items")
        
        for item in data[:3]:
            sku = item.get("sku")
            title = item.get("analysis", {}).get("metadata", {}).get("title", "")
            status = item.get("analysis", {}).get("status", "")
            print(f"\n  SKU {sku}: {title}")
            print(f"    Status: {status}")
        
        if len(data) > 3:
            print(f"\n  ... and {len(data) - 3} more items")
        
        return data
    except Exception as e:
        print(f"Error reading metadata: {e}")
        return None

# Example:
# metadata = read_generated_metadata()